In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# デバイスの設定
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# モデルのパス
model_path = '/home/nishioka/GNN/GNNmodel/10classmodel/GCNModel_20241019_131536best_model_fold_5.pth'

# モデル定義（学習時と同じ構造）
class GCNModel(torch.nn.Module):
    def __init__(self, hidden_channels=128, num_classes=10):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(4, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels * 2)
        self.conv3 = GCNConv(hidden_channels * 2, hidden_channels)
        self.fc = torch.nn.Linear(hidden_channels, num_classes)
        self.dropout = torch.nn.Dropout(p=0.2)

    def forward(self, data):
        x, edge_index = data.x.to(device), data.edge_index.to(device)
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv3(x, edge_index))
        x = self.fc(x)
        return F.log_softmax(x, dim=1)

# データのパス
standardized_data_folder = '/home/nishioka/GNN/Defect_4x4_Normalized1'
label_data_folder = '/home/nishioka/GNN/Defect10Class_OneHot_test1'

# 座標データのロード
x_coords = np.load('/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy')[:3654]
y_coords = np.load('/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy')[:3654]
z_coords = np.load('/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy')[:3654]

# エッジ情報のロード
edges = np.load('/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy')
edge_index = torch.tensor(edges.T, dtype=torch.long)

# データ準備関数
def prepare_data(pairs):
    data_list = []
    for idx, (data_file, label_file) in enumerate(pairs):
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        # データとラベルのロード
        values = np.load(data_file_path)[:3654]
        label = np.load(label_file_path)[:3654]

        # ラベルをクラスインデックスに変換（ワンホットエンコーディングの場合）
        label = label.argmax(axis=1)

        # ノード特徴量の作成
        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        x = torch.tensor(node_features, dtype=torch.float)
        y = torch.tensor(label, dtype=torch.long).squeeze()

        data = Data(x=x, edge_index=edge_index, y=y)
        data_list.append(data)
    return data_list

# データとラベルファイルの取得
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith('Normalized1_Defect4x4_ELNOD')]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith('Defect10Class_L')]
pairs = list(zip(data_files, label_files))

# 全データを準備
dataset = prepare_data(pairs)

# データローダーの作成
loader = DataLoader(dataset, batch_size=1, shuffle=False)

# モデルのロード
model = GCNModel(hidden_channels=32, num_classes=10).to(device)
model.load_state_dict(torch.load(model_path))
model.eval()

# 予測と正解ラベルの収集
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)
        out = model(batch)
        pred = out.argmax(dim=1).cpu().numpy()
        label = batch.y.view(-1).cpu().numpy()
        all_preds.extend(pred)
        all_labels.extend(label)

# 結果の表示
print('Classification Report:')
print(classification_report(all_labels, all_preds, digits=4))

# 混同行列の表示
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()
